# graph-vul: Qwen3-0.6B Fine-tuning on Google Colab

This notebook runs the function-level PrimeVul fine-tuning pipeline using Qwen3-0.6B as the code backbone, with Google Drive for persistent storage.

The split contract comes from `func_splits/assignments.csv`. In **project-disjoint** mode (default), all five splits (`train`, `val`, `test`, `unseen_val`, `unseen_test`) live in a single `assignments.csv`. In **official** mode, primary splits are in `assignments.csv` and derived unseen-CWE splits are in a separate `unseen_assignments.csv`.

Optionally, set `USE_GRAPH = True` to fuse a lightweight RGCN graph embedding
(AST+CFG structure) alongside the code and CWE embeddings.

## Partitioned Training

Set `PARTITION` in the config cell to train on a token-length partition:
- `"other"` (default) — functions with 65+ tokens
- `"short"` — functions with 0-64 tokens
- `None` — full dataset (all functions)

When using a partition, the notebook reads from `data/partitions/<name>/func_subset.csv`
and `data/partitions/<name>/splits/assignments.csv` instead of the top-level paths.

## Prerequisites

Before running, upload the following to **My Drive/graph-vul/data/**:

```
My Drive/
  graph-vul/
    data/
      func_subset.csv
      func_splits/
        assignments.csv
      partitions/              # only if using PARTITION
        short/
          func_subset.csv
          splits/
            assignments.csv
        other/
          func_subset.csv
          splits/
            assignments.csv
      processed/
        cwe_embeddings/
          <one .pt file for every CWE referenced by func_subset.csv>
        graph_artifacts.tar.gz  # only if USE_GRAPH = True
```

Create the graph archive locally with:

```bash
tar -czf data/processed/graph_artifacts.tar.gz -C data/processed graph_artifacts
```

The notebook extracts `processed/graph_artifacts.tar.gz` into local Colab storage
under `/content/graph_artifacts_cache/graph_artifacts` and symlinks
`/content/graph-vul/data/processed/graph_artifacts` to that local copy. This keeps
training reads off the Drive mount while preserving the existing repo `data/` path layout.

The notebook validates that:
- `assignments.csv` contains `train`, `val`, and `test`
- unseen splits (`unseen_val`, `unseen_test`) exist in either `assignments.csv` (project-disjoint mode) or a separate `unseen_assignments.csv` (official mode) — or warns if absent
- every CWE referenced by `func_subset.csv` has a matching embedding file in
  `processed/cwe_embeddings/`
- (when `USE_GRAPH = True`) `processed/graph_artifacts.tar.gz` exists, or falls back to an extracted `processed/graph_artifacts/` directory if present

The `hf_cache/` directory is created automatically on first run to cache the
`Qwen/Qwen3-0.6B` model weights (~1.2 GB). Subsequent runs skip the download.

**Outputs** (written to Drive):
- `data/func_finetune_model.ckpt` — best model checkpoint (selected by val_f1)
- `data/func_finetune_metrics.json` — train/val/test metrics plus `unseen_val`/`unseen_test` unseen-CWE metrics, including `*_0_64_f1` short-function metrics that are also logged to W&B

In [ ]:
import subprocess

gpu_info = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if gpu_info.returncode != 0:
    raise RuntimeError(
        "No GPU detected. Go to Runtime > Change runtime type > GPU."
    )
print(gpu_info.stdout)

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

DRIVE_PROJECT = "/content/drive/MyDrive/data/graph-vul"
DRIVE_DATA = f"{DRIVE_PROJECT}/data"
DRIVE_HF_CACHE = f"{DRIVE_PROJECT}/hf_cache"
DRIVE_GRAPH_ARCHIVE = f"{DRIVE_DATA}/processed/graph_artifacts.tar.gz"
LOCAL_GRAPH_ROOT = "/content/graph_artifacts_cache"
LOCAL_GRAPH_DIR = f"{LOCAL_GRAPH_ROOT}/graph_artifacts"

In [17]:
!mkdir -p {DRIVE_DATA}
!mkdir -p {DRIVE_HF_CACHE}

In [ ]:
import csv
from pathlib import Path

data_root = Path(DRIVE_DATA)

# Resolve partition-specific paths
try:
    _partition = PARTITION
except NameError:
    _partition = None

if _partition in ("short", "other"):
    subset_path = data_root / "partitions" / _partition / "func_subset.csv"
    assignments_path = data_root / "partitions" / _partition / "splits" / "assignments.csv"
    print(f"Using partition: {_partition}")
else:
    subset_path = data_root / "func_subset.csv"
    assignments_path = data_root / "func_splits" / "assignments.csv"
    print("Using full dataset (no partition)")

unseen_assignments_path = assignments_path.parent / "unseen_assignments.csv"
embeddings_dir = data_root / "processed" / "cwe_embeddings"

required_files = [subset_path, assignments_path, embeddings_dir]
missing = [str(path.relative_to(data_root)) for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError(
        f"Missing required paths in {DRIVE_DATA}:\n"
        + "\n".join(f"  - {path}" for path in missing)
    )

required_splits = {"train", "val", "test"}
with assignments_path.open(newline="") as handle:
    reader = csv.DictReader(handle)
    observed_splits = {row["split"] for row in reader if row.get("split")}

missing_splits = sorted(required_splits - observed_splits)
if missing_splits:
    raise ValueError(
        "assignments.csv is missing required splits:\n"
        + "\n".join(f"  - {split}" for split in missing_splits)
    )

# Unseen splits: check assignments.csv first (project-disjoint mode),
# fall back to unseen_assignments.csv (official mode)
required_unseen_splits = {"unseen_val", "unseen_test"}
unseen_in_main = required_unseen_splits.issubset(observed_splits)

if unseen_in_main:
    print("Unseen splits found in assignments.csv (project-disjoint mode)")
elif unseen_assignments_path.exists():
    with unseen_assignments_path.open(newline="") as handle:
        reader = csv.DictReader(handle)
        observed_unseen_splits = {row["split"] for row in reader if row.get("split")}
    missing_unseen = sorted(required_unseen_splits - observed_unseen_splits)
    if missing_unseen:
        raise ValueError(
            "unseen_assignments.csv is missing required splits:\n"
            + "\n".join(f"  - {split}" for split in missing_unseen)
        )
    print("Unseen splits found in unseen_assignments.csv (official mode)")
else:
    print("WARNING: No unseen splits found. Training will use train/val/test only.")

with subset_path.open(newline="") as handle:
    reader = csv.DictReader(handle)
    cwe_ids = sorted(
        {
            cwe_id.strip()
            for row in reader
            for cwe_id in [row.get("cwe_id") or ""]
            if cwe_id.strip()
        }
    )

missing_embeddings = [cwe_id for cwe_id in cwe_ids if not (embeddings_dir / f"{cwe_id}.pt").exists()]
if missing_embeddings:
    raise FileNotFoundError(
        "Missing CWE embeddings for func_subset.csv entries:\n"
        + "\n".join(f"  - processed/cwe_embeddings/{cwe_id}.pt" for cwe_id in missing_embeddings)
    )

print(f"Validated dataset root: {data_root}")
print(f"Found {len(cwe_ids)} CWE embedding files referenced by func_subset.csv.")
print(f"Observed primary splits: {sorted(observed_splits)}")

In [ ]:
# Validate graph artifacts (only checked when USE_GRAPH is enabled).
# This cell reads USE_GRAPH from the config cell below — re-run if you change it.
try:
    _use_graph = USE_GRAPH
except NameError:
    _use_graph = False

if _use_graph:
    graph_dir = data_root / "processed" / "graph_artifacts"
    graph_archive = Path(DRIVE_GRAPH_ARCHIVE)

    if graph_archive.exists():
        print(f"Graph mode enabled: found archive {graph_archive}")
    elif graph_dir.exists():
        node_vocab = graph_dir / "node_type_vocab.json"
        edge_vocab = graph_dir / "edge_type_vocab.json"
        missing_graph = []
        if not node_vocab.exists():
            missing_graph.append(str(node_vocab.relative_to(data_root)))
        if not edge_vocab.exists():
            missing_graph.append(str(edge_vocab.relative_to(data_root)))
        if missing_graph:
            raise FileNotFoundError(
                f"USE_GRAPH=True but missing graph artifacts in {DRIVE_DATA}:\n"
                + "\n".join(f"  - {p}" for p in missing_graph)
            )
        sample_graph = next(graph_dir.glob("*.pt"), None)
        if sample_graph is None:
            raise FileNotFoundError(
                f"USE_GRAPH=True but no graph .pt files were found in {graph_dir}"
            )
        print(f"Graph mode enabled: using extracted artifacts in {graph_dir}")
    else:
        raise FileNotFoundError(
            f"USE_GRAPH=True but neither {graph_dir} nor {graph_archive} exists"
        )
else:
    print("Graph mode disabled (USE_GRAPH=False). Skipping graph validation.")

In [20]:
import os

os.makedirs(DRIVE_HF_CACHE, exist_ok=True)
os.environ["HF_HOME"] = DRIVE_HF_CACHE

In [ ]:
from google.colab import userdata

github_token = userdata.get('GITHUB_TOKEB_GRAPH_VUL')
if not github_token:
    raise ValueError('Missing GITHUB_TOKEB_GRAPH_VUL in Colab secrets')

!git clone https://{github_token}@github.com/noamdwc/graph-vul.git /content/graph-vul
%pip install -q pytorch-lightning transformers sentencepiece pandas numpy \
    scikit-learn torchmetrics wandb torch-geometric

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

repo_root = Path("/content/graph-vul")
repo_data = repo_root / "data"
repo_processed = repo_data / "processed"
drive_data = Path(DRIVE_DATA)
drive_processed = drive_data / "processed"
drive_graph_dir = drive_processed / "graph_artifacts"
drive_graph_archive = Path(DRIVE_GRAPH_ARCHIVE)
local_graph_dir = Path(LOCAL_GRAPH_DIR)

def reset_path(path: Path) -> None:
    if path.is_symlink() or path.is_file():
        path.unlink()
    elif path.is_dir():
        shutil.rmtree(path)

def symlink_path(src: Path, dst: Path) -> None:
    dst.parent.mkdir(parents=True, exist_ok=True)
    reset_path(dst)
    dst.symlink_to(src, target_is_directory=src.is_dir())

reset_path(repo_data)
repo_processed.mkdir(parents=True, exist_ok=True)

for relative_path in ["func_subset.csv", "func_splits", "partitions"]:
    src = drive_data / relative_path
    if src.exists():
        symlink_path(src, repo_data / relative_path)

symlink_path(drive_processed / "cwe_embeddings", repo_processed / "cwe_embeddings")

if drive_graph_archive.exists():
    local_graph_dir.parent.mkdir(parents=True, exist_ok=True)
    if not local_graph_dir.exists():
        print(f"Extracting {drive_graph_archive} to {local_graph_dir.parent} ...")
        members = subprocess.run(
            ["tar", "-tzf", str(drive_graph_archive)],
            capture_output=True,
            text=True,
            check=True,
        ).stdout.splitlines()
        members = [member.strip() for member in members if member.strip()]
        if not members:
            raise ValueError(f"Archive is empty: {drive_graph_archive}")
        graph_member = next((member for member in members if "graph_artifacts/" in member), None)
        if graph_member is None:
            raise ValueError(f"Archive does not contain graph_artifacts: {drive_graph_archive}")
        strip_components = Path(graph_member).parts.index("graph_artifacts")
        reset_path(local_graph_dir)
        subprocess.run(
            [
                "tar",
                "-xzf",
                str(drive_graph_archive),
                "-C",
                str(local_graph_dir.parent),
                f"--strip-components={strip_components}",
            ],
            check=True,
        )
        node_vocab = local_graph_dir / "node_type_vocab.json"
        edge_vocab = local_graph_dir / "edge_type_vocab.json"
        if not node_vocab.exists() or not edge_vocab.exists():
            raise ValueError(
                "Extracted archive does not contain node_type_vocab.json and edge_type_vocab.json"
            )
    else:
        print(f"Reusing local graph cache at {local_graph_dir}")
    symlink_path(local_graph_dir, repo_processed / "graph_artifacts")
elif drive_graph_dir.exists():
    symlink_path(drive_graph_dir, repo_processed / "graph_artifacts")
    print("Graph artifacts will be read directly from Drive (slower for training).")
else:
    print("No graph artifacts found during setup. This is fine if USE_GRAPH stays False.")

assert Path(repo_data, "func_subset.csv").exists(), "Dataset wiring failed: func_subset.csv missing"
assert Path(repo_processed, "cwe_embeddings").exists(), "Dataset wiring failed: cwe embeddings missing"
print(f"Prepared repo data tree at {repo_data}")
print(f"Primary data source: {drive_data}")
if (repo_processed / "graph_artifacts").exists():
    print(f"Graph artifacts path: {repo_processed / 'graph_artifacts'}")

## Weights & Biases

The training script logs to WandB. Run the cell below to authenticate.
To skip WandB logging, uncomment the `WANDB_MODE` line instead.

In [ ]:
import os
from google.colab import userdata
os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
!wandb login

## Training Configuration

Adjust hyperparameters in the cell below. Defaults match `graph_vul/config.py`.

| Parameter | Default | Notes |
|-----------|---------|-------|
| `NUM_EPOCHS` | 30 | Max epochs (early stopping will halt sooner) |
| `BATCH_SIZE` | 8 | Per-GPU batch size |
| `ENCODER_LR` | 2e-5 | Qwen3 backbone learning rate |
| `HEAD_LR` | 1e-3 | Classification head learning rate |
| `GRAD_ACCUM` | 8 | Effective batch = BATCH_SIZE x GRAD_ACCUM = 64 |
| `SEED` | 42 | |
| `USE_GRAPH` | False | Enable RGCN graph (AST+CFG) features only if graph artifacts were uploaded |
| `FUSION_LAYERS` | `list(range(28))` | All-layer gated residual fusion (28 = Qwen3-0.6B decoder depth) |
| `FINAL_GRAPH_CONCAT` | True | Late fusion: concatenate graph embedding before classifier |
| `PARTITION` | `"other"` | `None` = full dataset, `"short"` = 0-64 tokens, `"other"` = 65+ tokens |
| `FREEZE_ENCODER_EPOCHS` | 3 | Epochs to freeze encoder (head-only warmup) |
| `UNFREEZE_ENCODER_LR` | 5e-6 | Encoder LR after unfreezing (lower than `ENCODER_LR` to prevent forgetting) |
| `FOCAL_GAMMA` | 2.0 | Focal loss gamma — down-weights easy examples (0 = standard BCE) |
| `FOCAL_ALPHA` | 0.25 | Focal loss alpha — weight for positive class |
| `HEAD_DROPOUT` | 0.5 | Dropout in classification head (default config: 0.3) |
| `FREEZE_ENCODER_FULLY` | False | Keep encoder frozen for all epochs (head-only training) |
| `GRAPH_ARCHIVE_NAME` | `"graph_artifacts.tar.gz"` | Archive name expected under `data/processed/` when `USE_GRAPH=True` |

**OOM?** Reduce `BATCH_SIZE` to 4 and increase `GRAD_ACCUM` to 16 to keep effective batch size at 64.

In [ ]:
NUM_EPOCHS = 30
BATCH_SIZE = 16
ENCODER_LR = 2e-5
HEAD_LR = 1e-3
GRAD_ACCUM = 8
SEED = 42
USE_GRAPH = True
FUSION_LAYERS = list(range(28))  # 28 = number of decoder layers in Qwen3-0.6B; fuses graph embedding into every block.
FINAL_GRAPH_CONCAT = True
FREEZE_ENCODER_EPOCHS = 3
UNFREEZE_ENCODER_LR = 5e-6
GRAPH_ARCHIVE_NAME = "graph_artifacts.tar.gz"
DRIVE_GRAPH_ARCHIVE = f"{DRIVE_DATA}/processed/{GRAPH_ARCHIVE_NAME}"

# Anti-overfit settings
FOCAL_GAMMA = 2.0        # focal loss gamma (0 = standard BCE)
FOCAL_ALPHA = 0.25       # focal loss alpha (positive class weight)
HEAD_DROPOUT = 0.5        # higher dropout in head (default: 0.3)
FREEZE_ENCODER_FULLY = False  # True = head-only training (no encoder fine-tuning)

# Partition: None = full dataset, "short" = 0-64 tokens, "other" = 65+ tokens
PARTITION = "other"

In [ ]:
import logging
import sys
from pathlib import Path

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

sys.path.insert(0, "/content/graph-vul")

# Override config paths when using a partition
import graph_vul.config as _cfg

try:
    _partition = PARTITION
except NameError:
    _partition = None

if _partition in ("short", "other"):
    _cfg.FUNC_SUBSET_CSV = _cfg.DATA_DIR / "partitions" / _partition / "func_subset.csv"
    _cfg.FUNC_SPLITS_DIR = _cfg.DATA_DIR / "partitions" / _partition / "splits"
    print(f"Config overridden for partition '{_partition}':")
    print(f"  FUNC_SUBSET_CSV = {_cfg.FUNC_SUBSET_CSV}")
    print(f"  FUNC_SPLITS_DIR = {_cfg.FUNC_SPLITS_DIR}")

checkpoint_path = Path(DRIVE_DATA) / "func_finetune_model.ckpt"
metrics_path = Path(DRIVE_DATA) / "func_finetune_metrics.json"

from graph_vul.train_finetune import train

train(
    num_epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    encoder_lr=ENCODER_LR,
    head_lr=HEAD_LR,
    grad_accum=GRAD_ACCUM,
    seed=SEED,
    use_graph=USE_GRAPH,
    fusion_layers=FUSION_LAYERS,
    final_graph_concat=FINAL_GRAPH_CONCAT,
    checkpoint_path=checkpoint_path,
    metrics_path=metrics_path,
    freeze_encoder_epochs=FREEZE_ENCODER_EPOCHS,
    unfreeze_encoder_lr=UNFREEZE_ENCODER_LR,
    focal_gamma=FOCAL_GAMMA,
    focal_alpha=FOCAL_ALPHA,
    head_dropout=HEAD_DROPOUT,
    freeze_encoder_fully=FREEZE_ENCODER_FULLY,
)

In [ ]:
import json
from pathlib import Path

metrics_path = Path(DRIVE_DATA) / "func_finetune_metrics.json"
ckpt_path = Path(DRIVE_DATA) / "func_finetune_model.ckpt"

if metrics_path.exists():
    with open(metrics_path) as f:
        metrics = json.load(f)

    groups = {
        "Primary Split Metrics": ["train", "val", "test"],
        "Derived Unseen-CWE Metrics": ["unseen_val", "unseen_test"],
    }
    priority_suffixes = ["accuracy", "precision", "recall", "f1", "0_64_f1"]

    def ordered_keys(prefix):
        present = [k for k in metrics if k.startswith(f"{prefix}_")]
        ordered = [f"{prefix}_{suffix}" for suffix in priority_suffixes if f"{prefix}_{suffix}" in metrics]
        extras = sorted(k for k in present if k not in ordered)
        return ordered + extras

    for heading, prefixes in groups.items():
        print(f"=== {heading} ===")
        for prefix in prefixes:
            keys = ordered_keys(prefix)
            if keys:
                vals = "  ".join(f"{k}: {metrics[k]:.4f}" for k in keys)
                print(f"  {vals}")
                short_key = f"{prefix}_0_64_f1"
                if short_key in metrics:
                    print(f"    short functions (0-64 tokens) F1: {metrics[short_key]:.4f}")
            else:
                print(f"  {prefix}: no metrics found")
        print()
else:
    print(f"Metrics file not found at {metrics_path}")

if ckpt_path.exists():
    size_mb = ckpt_path.stat().st_size / (1024 * 1024)
    print(f"\nCheckpoint saved: {ckpt_path} ({size_mb:.1f} MB)")
else:
    print(f"\nCheckpoint not found at {ckpt_path}")